In [ ]:
import os
import glob
import numpy as np
import jax
import jax.numpy as jnp
from flax import linen as nn
import pennylane as qml
import optax
from torch.utils.data import Dataset, DataLoader # We use Torch's efficient loader

In [ ]:
# ==========================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==========================================
DATA_PATH = "./Sen2Fire"  # <--- CHANGE THIS to your extracted folder path
BATCH_SIZE = 8            # Paper used 8
LEARNING_RATE = 1e-4      # Paper used 1e-4
IMG_SIZE = 512            # Native size of Sen2Fire

In [ ]:
# ==========================================
# 2. DATA LOADER (Specific to Sen2Fire)
# ==========================================
class Sen2FireDataset(Dataset):
    def __init__(self, root_dir, split='train'):
        # Paper splits: Train (Area 1,2), Val (Area 3), Test (Area 4)
        # For simplicity, we just load all .npz files found in the dir
        self.files = glob.glob(os.path.join(root_dir, "**/*.npz"), recursive=True)
        print(f"Found {len(self.files)} files.")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # Load the numpy archive
        data = np.load(self.files[idx])
        
        # Keys in Sen2Fire are typically 'image' and 'label' or similar. 
        # We assume standard structure based on the paper description.
        # Sen2Fire Image Shape: (13, 512, 512) or (512, 512, 13)
        img = data['image'] 
        mask = data['label']
        
        # --- STRATEGY: SWIR COMPOSITE ---
        # The paper says SWIR composite uses bands B12, B8, B4.
        # Indices (0-based): B12=11, B8=7, B4=3
        indices = [11, 7, 3]
        
        # Extract only these bands
        # Assuming input is (Channels, H, W). If (H, W, C), change axis.
        if img.shape[0] == 13: 
            img = img[indices, :, :] # (3, 512, 512)
            img = np.transpose(img, (1, 2, 0)) # To (512, 512, 3) for JAX
        else:
            img = img[:, :, indices]

        # Normalize! Sentinel data is 16-bit (0-10000+). We map to 0-1.
        img = img.astype(np.float32) / 10000.0
        img = np.clip(img, 0, 1) # Clip outliers
        
        # Prepare Mask (Binary 0 or 1)
        mask = mask.astype(np.float32)
        if len(mask.shape) == 2:
            mask = np.expand_dims(mask, axis=-1) # (512, 512, 1)

        return img, mask

# Helper to collate batch for JAX
def numpy_collate(batch):
    if isinstance(batch[0], np.ndarray):
        return np.stack(batch)
    elif isinstance(batch[0], (tuple,list)):
        transposed = zip(*batch)
        return [numpy_collate(samples) for samples in transposed]
    else:
        return np.array(batch)

In [ ]:
# ==========================================
# 3. QUANTUM LAYER (Bottleneck)
# ==========================================
n_qubits = 4
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="jax")
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(n_qubits)]

class QuantumLayer(nn.Module):
    @nn.compact
    def __call__(self, x):
        weights = self.param('weights', nn.initializers.uniform(scale=0.1), (n_layers, n_qubits))
        # Vectorize over Batch, Height, Width
        q_node_vmap = jax.vmap(jax.vmap(jax.vmap(quantum_circuit, in_axes=(0, None)), in_axes=(0, None)), in_axes=(0, None))
        q_out = q_node_vmap(x, weights)
        return jnp.stack(q_out, axis=-1)

In [ ]:
# ==========================================
# 4. DEEP HYBRID U-NET (512x512 -> 32x32)
# ==========================================
# We reuse the DoubleConv from previous step (with GroupNorm!)
class DoubleConv(nn.Module):
    out_channels: int
    @nn.compact
    def __call__(self, x):
        x = nn.Conv(self.out_channels, (3, 3), padding='SAME')(x)
        x = nn.GroupNorm()(x)
        x = nn.relu(x)
        x = nn.Conv(self.out_channels, (3, 3), padding='SAME')(x)
        x = nn.GroupNorm()(x)
        x = nn.relu(x)
        return x

class DeepHybridUNet(nn.Module):
    out_channels: int

    @nn.compact
    def __call__(self, x):
        # --- ENCODER (Contracting) ---
        # 512 -> 256
        c1 = DoubleConv(32)(x)
        p1 = nn.max_pool(c1, window_shape=(2, 2), strides=(2, 2))
        
        # 256 -> 128
        c2 = DoubleConv(64)(p1)
        p2 = nn.max_pool(c2, window_shape=(2, 2), strides=(2, 2))

        # 128 -> 64
        c3 = DoubleConv(128)(p2)
        p3 = nn.max_pool(c3, window_shape=(2, 2), strides=(2, 2))

        # 64 -> 32
        c4 = DoubleConv(256)(p3)
        p4 = nn.max_pool(c4, window_shape=(2, 2), strides=(2, 2))

        # --- QUANTUM BOTTLENECK (at 32x32 resolution) ---
        # We are now at 32x32 size. This is safe for the quantum layer.
        
        # Compress features to 4 for the quantum circuit
        q_in = nn.Conv(4, (1, 1))(p4)
        q_in = nn.relu(q_in)
        
        # Run Quantum Circuit
        q_out = QuantumLayer()(q_in)
        q_out = nn.relu(q_out)
        
        # Expand back
        bottleneck = nn.Conv(512, (1, 1))(q_out)
        bottleneck = nn.relu(bottleneck)

        # --- DECODER (Expansive) ---
        # Up 1: 32 -> 64
        u1 = nn.ConvTranspose(256, (2, 2), strides=(2, 2), padding='SAME')(bottleneck)
        u1 = jnp.concatenate([u1, c4], axis=-1)
        c5 = DoubleConv(256)(u1)

        # Up 2: 64 -> 128
        u2 = nn.ConvTranspose(128, (2, 2), strides=(2, 2), padding='SAME')(c5)
        u2 = jnp.concatenate([u2, c3], axis=-1)
        c6 = DoubleConv(128)(u2)

        # Up 3: 128 -> 256
        u3 = nn.ConvTranspose(64, (2, 2), strides=(2, 2), padding='SAME')(c6)
        u3 = jnp.concatenate([u3, c2], axis=-1)
        c7 = DoubleConv(64)(u3)

        # Up 4: 256 -> 512
        u4 = nn.ConvTranspose(32, (2, 2), strides=(2, 2), padding='SAME')(c7)
        u4 = jnp.concatenate([u4, c1], axis=-1)
        c8 = DoubleConv(32)(u4)

        # Final Output
        return nn.Conv(self.out_channels, (1, 1))(c8)

In [ ]:
# ==========================================
# 5. TRAINING SETUP
# ==========================================
def train_sen2fire():
    # 1. Setup Data
    # NOTE: You must set DATA_PATH to your actual download folder
    if not os.path.exists(DATA_PATH):
        print(f"Please download Sen2Fire data to {DATA_PATH} first!")
        return

    dataset = Sen2FireDataset(DATA_PATH)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=numpy_collate)
    
    # 2. Setup Model
    model = DeepHybridUNet(out_channels=1)
    
    # Init params with dummy input
    dummy_x = jnp.ones((1, IMG_SIZE, IMG_SIZE, 3)) 
    key = jax.random.PRNGKey(0)
    variables = model.init(key, dummy_x)
    params = variables['params']
    
    # 3. Optimizer
    tx = optax.adam(learning_rate=LEARNING_RATE)
    opt_state = tx.init(params)
    
    # 4. Loss Function (Weighted for Class Imbalance)
    # Fire pixels are rare. We weight them higher.
    def loss_fn(params, x, y):
        logits = model.apply({'params': params}, x)
        # Sigmoid Cross Entropy
        loss = optax.sigmoid_binary_cross_entropy(logits, y)
        # Weight fire pixels (1) 10x more than background (0)
        weight = y * 10.0 + (1 - y) * 1.0 
        return jnp.mean(loss * weight)

    @jax.jit
    def train_step(params, opt_state, x, y):
        loss, grads = jax.value_and_grad(loss_fn)(params, x, y)
        updates, new_opt_state = tx.update(grads, opt_state)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt_state, loss

    # 5. Loop
    print(f"Starting training on Sen2Fire (SWIR Composite)...")
    
    for epoch in range(3):
        for i, (x_batch, y_batch) in enumerate(dataloader):
            x_batch = jnp.array(x_batch)
            y_batch = jnp.array(y_batch)
            
            params, opt_state, loss = train_step(params, opt_state, x_batch, y_batch)
            
            if i % 10 == 0:
                print(f"Epoch {epoch} | Batch {i} | Loss: {loss:.4f}")

In [ ]:
train_sen2fire()